# 3. Explainability & Analysis

Grad-CAM visualizations, filter analysis, error analysis, and conclusions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
from pathlib import Path

from src.utils import load_config, get_device
from src.utils.visualization import set_style, plot_predictions_vs_actual
from src.data import create_dataloaders
from src.models import WatchPriceCNN
from src.evaluation import predict, compute_metrics, print_metrics
from src.explainability import generate_gradcam, plot_gradcam_grid, visualize_first_layer_filters

set_style()
config = load_config('../configs/base.yaml')
config['data']['metadata_path'] = '../data/metadata.csv'
config['data']['root_dir'] = '../data/raw'
device = get_device(config)

## 3.1 Load best model

In [ ]:
checkpoint = torch.load('../outputs/checkpoints/best_model.pt', map_location=device, weights_only=False)
model = WatchPriceCNN(config['model'])
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
print(f'Loaded from epoch {checkpoint["epoch"]} (val_loss={checkpoint["val_loss"]:.4f})')

## 3.2 Test set evaluation

In [ ]:
_, _, test_loader = create_dataloaders(config)
y_true, y_pred = predict(model, test_loader, device)
metrics = compute_metrics(y_true, y_pred, log_transformed=True)
print_metrics(metrics)

In [ ]:
y_true_dollar = np.expm1(y_true)
y_pred_dollar = np.expm1(y_pred)
plot_predictions_vs_actual(y_true_dollar, y_pred_dollar)

## 3.3 Grad-CAM analysis

Which regions of the watch image drive the price prediction?

In [ ]:
images, targets = next(iter(test_loader))
n = min(20, len(images))
images, targets = images[:n], targets[:n]

cams = generate_gradcam(model, images, device=device)

with torch.no_grad():
    preds = model(images.to(device)).cpu().numpy()

plot_gradcam_grid(
    images, cams,
    predictions=np.expm1(preds),
    actuals=np.expm1(targets.numpy()),
)

## 3.4 First layer filters

In [ ]:
visualize_first_layer_filters(model)

## 3.5 Error analysis

Where does the model fail? Analyze by brand, price range, and image characteristics.

In [ ]:
# TODO: Error analysis by brand, price range
# - Worst predictions (largest absolute error)
# - Error distribution by price bin
# - Error distribution by brand
# - Are there visual patterns in the worst predictions?

## 3.6 Conclusions

### Key findings
- _TBD_

### Limitations
- _TBD_

### Possible improvements
- _TBD_